# Remote Cleanup 02: `device.cleanup()` and Constructor Auto-Cleanup

This notebook covers two manual flows:

1. Explicit global cleanup with `device.cleanup()`.
2. Leaving state behind, restarting the kernel, and then checking whether the next device construction cleans it.

For the stale-state seed section, set `USE_PROBED_DEVICE = False` so you can create `RemoteDevice(auto_cleanup=False)` explicitly.

In [ ]:
import json
import os
from pathlib import Path

import grpc
import numpy as np

REMOTE_IP = os.environ.get("PYNQ_REMOTE_DEVICES", "192.168.2.197").split(",")[0].strip()
USE_PROBED_DEVICE = False
OVERLAY_NAME = "resizer.xsa"
STATE_FILE = Path("/tmp/pynq_remote_constructor_autocleanup_state.json")

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "tests" / OVERLAY_NAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not find tests/{OVERLAY_NAME} from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
OVERLAY_PATH = REPO_ROOT / "tests" / OVERLAY_NAME

os.environ["PYNQ_REMOTE_DEVICES"] = REMOTE_IP

from pynq import GPIO, Overlay
from pynq.pl_server.device import Device
from pynq.pl_server.remote_device import RemoteDevice
from pynq.remote import buffer_pb2, gpio_pb2, mmio_pb2

def get_device(auto_cleanup: bool = True):
    if USE_PROBED_DEVICE:
        devices = [d for d in Device.devices if isinstance(d, RemoteDevice)]
        if not devices:
            raise RuntimeError("No probed RemoteDevice found. Check PYNQ_REMOTE_DEVICES and connectivity.")
        if auto_cleanup is False:
            raise RuntimeError("Set USE_PROBED_DEVICE = False when you need RemoteDevice(auto_cleanup=False).")
        return devices[0]
    return RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=auto_cleanup)

print(f"REMOTE_IP={REMOTE_IP}")
print(f"USE_PROBED_DEVICE={USE_PROBED_DEVICE}")
print(f"OVERLAY_PATH={OVERLAY_PATH}")
print(f"STATE_FILE={STATE_FILE}")

## Part A: Explicit `device.cleanup()`

In [ ]:
device = get_device(auto_cleanup=True)
overlay = Overlay(str(OVERLAY_PATH), device=device)
base_addr = overlay.ip_dict["resize_accel_0"]["phys_addr"]

remote_mmio = device.mmap(base_addr, 0x1000)
mmio_id = remote_mmio.mmio_id
remote_mmio.read(0)

remote_buffer = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
buffer_id = remote_buffer.buffer_id
remote_buffer[:] = np.arange(16, dtype=np.uint32)
remote_buffer.flush()

remote_gpio = None
gpio_id = None
gpio_path = None
base_path = GPIO.get_gpio_base_path(device=device)
npins = GPIO.get_gpio_npins(device=device)
if base_path and npins:
    gpio_pin = GPIO.get_gpio_pin(0, device=device)
    gpio_path = f"/sys/class/gpio/gpio{gpio_pin}"
    if not device.exists_file(gpio_path).exists:
        remote_gpio = GPIO(gpio_pin, "in", device=device)
        remote_gpio.read()
        gpio_id = remote_gpio._gpio_id
    else:
        print(f"Skipping GPIO because {gpio_path} is already exported.")
else:
    print("Skipping GPIO because Linux sysfs GPIO is not available.")

print(f"MMIO id:   {mmio_id}")
print(f"Buffer id: {buffer_id}")
if gpio_id is not None:
    print(f"GPIO id:   {gpio_id} at {gpio_path}")

input("Check the board logs for resource creation, then press Enter to call device.cleanup().")
response = device.cleanup()
print(response)
input("Check the board logs for the Cleanup Request, then press Enter to verify the old objects and IDs are stale.")

for description, callable_ in [
    ("stale MMIO object", lambda: remote_mmio.read(0)),
    ("stale buffer object", lambda: remote_buffer.physical_address),
]:
    try:
        callable_()
        raise AssertionError(f"{description} still worked after device.cleanup().")
    except (grpc.RpcError, RuntimeError):
        print(f"{description} is stale as expected.")

try:
    device._stub["mmio"].read(
        mmio_pb2.ReadRequest(mmio_id=mmio_id, offset=0, length=4, word_order="little")
    )
    raise AssertionError(f"MMIO id {mmio_id} still exists after device.cleanup().")
except (grpc.RpcError, RuntimeError):
    print(f"MMIO id {mmio_id} is invalid as expected.")

try:
    device._stub["buffer"].physical_address(buffer_pb2.AddressRequest(buffer_id=buffer_id))
    raise AssertionError(f"Buffer id {buffer_id} still exists after device.cleanup().")
except (grpc.RpcError, RuntimeError):
    print(f"Buffer id {buffer_id} is invalid as expected.")

if gpio_id is not None:
    try:
        remote_gpio.read()
        raise AssertionError("stale GPIO object still worked after device.cleanup().")
    except (grpc.RpcError, RuntimeError):
        print("stale GPIO object is stale as expected.")
    try:
        device._stub["gpio"].read(gpio_pb2.GpioReadRequest(gpio_id=gpio_id))
        raise AssertionError(f"GPIO id {gpio_id} still exists after device.cleanup().")
    except (grpc.RpcError, RuntimeError):
        print(f"GPIO id {gpio_id} is invalid as expected.")
    assert device.exists_file(gpio_path).exists is False
    print(f"GPIO sysfs path {gpio_path} has been removed.")

## Part B: Seed stale state, then restart the kernel

The next cell is meant to be run with `USE_PROBED_DEVICE = False`.

It creates remote resources with `auto_cleanup=False` and writes their IDs to `STATE_FILE`. After the cell finishes, use **Kernel -> Restart Kernel**. Then run the final cell in this notebook.

The post-restart check will first probe those old IDs with `RemoteDevice(auto_cleanup=False)`. If they are still alive, the restart was effectively hard enough to orphan resources. It will then construct `RemoteDevice(auto_cleanup=True)` and confirm that constructor auto-cleanup removes them.

In [ ]:
seed_device = get_device(auto_cleanup=False)
seed_overlay = Overlay(str(OVERLAY_PATH), device=seed_device)
seed_base_addr = seed_overlay.ip_dict["resize_accel_0"]["phys_addr"]

seed_mmio = seed_device.mmap(seed_base_addr, 0x1000)
seed_mmio.read(0)
seed_buffer = seed_device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
seed_buffer[:] = np.arange(16, dtype=np.uint32)
seed_buffer.flush()

seed_gpio = None
seed_gpio_id = None
seed_gpio_path = None
base_path = GPIO.get_gpio_base_path(device=seed_device)
npins = GPIO.get_gpio_npins(device=seed_device)
if base_path and npins:
    gpio_pin = GPIO.get_gpio_pin(0, device=seed_device)
    seed_gpio_path = f"/sys/class/gpio/gpio{gpio_pin}"
    if not seed_device.exists_file(seed_gpio_path).exists:
        seed_gpio = GPIO(gpio_pin, "in", device=seed_device)
        seed_gpio.read()
        seed_gpio_id = seed_gpio._gpio_id
    else:
        print(f"Skipping GPIO because {seed_gpio_path} is already exported.")
else:
    print("Skipping GPIO because Linux sysfs GPIO is not available.")

seed_state = {
    "mmio_id": seed_mmio.mmio_id,
    "buffer_id": seed_buffer.buffer_id,
    "gpio_id": seed_gpio_id,
    "gpio_path": seed_gpio_path,
    "base_addr": seed_base_addr,
}
STATE_FILE.write_text(json.dumps(seed_state, indent=2))
print(seed_state)
input("Check the board logs for the live seed resources, then restart the kernel before running the next cell.")

In [ ]:
state = json.loads(STATE_FILE.read_text())
print(state)

probe_device = RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=False)

def raw_id_is_live():
    results = {}
    try:
        probe_device._stub["mmio"].read(
            mmio_pb2.ReadRequest(mmio_id=state["mmio_id"], offset=0, length=4, word_order="little")
        )
        results["mmio"] = True
    except (grpc.RpcError, RuntimeError):
        results["mmio"] = False
    try:
        probe_device._stub["buffer"].physical_address(
            buffer_pb2.AddressRequest(buffer_id=state["buffer_id"])
        )
        results["buffer"] = True
    except (grpc.RpcError, RuntimeError):
        results["buffer"] = False
    if state["gpio_id"] is not None:
        try:
            probe_device._stub["gpio"].read(gpio_pb2.GpioReadRequest(gpio_id=state["gpio_id"]))
            results["gpio"] = True
        except (grpc.RpcError, RuntimeError):
            results["gpio"] = False
    return results

pre_cleanup_results = raw_id_is_live()
print("Before constructor auto-cleanup:", pre_cleanup_results)
if state["gpio_path"] is not None:
    print("GPIO path exists before constructor auto-cleanup:", probe_device.exists_file(state["gpio_path"]).exists)

input("Check the board logs for the pre-cleanup probe, then press Enter to construct a fresh auto_cleanup device.")

cleanup_device = RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=True)
input("Constructor auto-cleanup has now run. Check the board logs, then press Enter to verify that the old IDs are invalid.")

for resource_name, is_live in raw_id_is_live().items():
    assert is_live is False, f"{resource_name} is still live after constructor auto-cleanup"
    print(f"{resource_name} is invalid after constructor auto-cleanup as expected.")
if state["gpio_path"] is not None:
    assert cleanup_device.exists_file(state["gpio_path"]).exists is False
    print(f"GPIO sysfs path {state['gpio_path']} has been removed.")